# Experiment: Colab Bootstrap And Smoke Test

Objective:
- Bootstrap the repo in Colab.
- Sync the assistant-axis source inputs.
- Run a tiny pipeline smoke test before moving the full rollout and activation jobs to Vast.ai.


In [ ]:
# Colab bootstrap
from __future__ import annotations

from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

REPO_URL = 'https://github.com/mahadikprasad15/Deception-Honesty-Axis.git'
REPO_DIR = Path('/content/Deception-Honesty-Axis')
REPO_DIR


## Plan

- Clone the repo and install dependencies.
- Create a temporary smoke-test config with `1 instruction x 2 questions`.
- Sync source files, generate a tiny rollout set, and audit coverage.
- Use the same scripts later on Vast.ai with the real `2 x 100` config.


In [ ]:
# Clone and install once per runtime
%cd /content
!test -d Deception-Honesty-Axis || git clone {REPO_URL}
%cd /content/Deception-Honesty-Axis
!python3 -m pip install -q -r requirements.txt
!python3 -m pip install -q -e .


## Results

- Smoke-test checklist:
- Source files synced into `data/`
- Rollout corpus created under `artifacts/corpora/...`
- Audit report written to `artifacts/corpora/.../meta/audit_report.json`
- If this passes, move the full `2 x 100` run to Vast.ai


In [ ]:
# Prepare a tiny smoke-test config and run the first stages
import json

config_path = Path('configs/experiments/deception_v1_llama32_3b.json')
smoke_config_path = Path('/tmp/deception_v1_smoke.json')
config = json.loads(config_path.read_text())
config['subset']['instruction_count'] = 1
config['subset']['question_count'] = 2
smoke_config_path.write_text(json.dumps(config, indent=2))

!PYTHONPATH=src python3 scripts/sync_source_data.py --config /tmp/deception_v1_smoke.json
!PYTHONPATH=src python3 scripts/generate_rollouts.py --config /tmp/deception_v1_smoke.json --batch-size 1
!PYTHONPATH=src python3 scripts/audit_corpus.py --config /tmp/deception_v1_smoke.json


## Next steps

- Run `scripts/extract_activations.py` on Vast.ai for the real `2 x 100` config.
- Push completed artifact trees to the HF dataset repo with `scripts/sync_hf_artifacts.py`.
- Build role vectors, then run PCA with the resulting vectors run directory after activation extraction finishes.
